In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import PIL.Image
import numpy as np

# Create required directories
os.makedirs("models", exist_ok=True)
os.makedirs("data/sample_images", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Dataset Loading & Preprocessing

# ResNet-18 expects 3 channels, 224x224 inputs, and ImageNet normalization
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# Stratified Validation Split (55,000 Train / 5,000 Val)
targets = full_train_dataset.targets.numpy()
train_idx, val_idx = train_test_split(
    np.arange(len(targets)), test_size=5000, stratify=targets, random_state=42
)

train_data = Subset(full_train_dataset, train_idx)
val_data = Subset(full_train_dataset, val_idx)

# Report Sizes
print(f"Data Splits -> Train: {len(train_data)} | Validation: {len(val_data)} | Test: {len(test_dataset)}")

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

Using device: cuda
Data Splits -> Train: 55000 | Validation: 5000 | Test: 10000


In [6]:
# 2. Build Transfer Learning Model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze early/middle layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head for 10 classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)
model = model.to(device)

In [7]:
# 3. Train Head (Using Caching Speed Trick)
print("\n[Phase 1] Extracting and caching backbone features...")
model.eval()
feature_extractor = nn.Sequential(*list(model.children())[:-1]).to(device)

def cache_features(dataloader):
    features, labels = [], []
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            feats = feature_extractor(inputs).squeeze()
            features.append(feats.cpu())
            labels.append(targets)
    return torch.cat(features), torch.cat(labels)

train_feats, train_lbls = cache_features(train_loader)
val_feats, val_lbls = cache_features(val_loader)

fast_train_loader = DataLoader(TensorDataset(train_feats, train_lbls), batch_size=128, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
epochs = 5

print(f"\n[Phase 2] Training classifier head (Batch Size: 128, LR: 0.001, Opt: Adam)")
model.fc.train()
for epoch in range(epochs):
    running_loss = 0.0
    for feats, labels in fast_train_loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model.fc(feats)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(fast_train_loader):.4f}")

# Evaluate Validation Accuracy
model.fc.eval()
with torch.no_grad():
    val_outputs = model.fc(val_feats.to(device))
    _, val_preds = torch.max(val_outputs, 1)
    val_acc = accuracy_score(val_lbls.numpy(), val_preds.cpu().numpy())

print(f"\nValidation Accuracy after Feature Extraction: {val_acc:.4f}")


[Phase 1] Extracting and caching backbone features...

[Phase 2] Training classifier head (Batch Size: 128, LR: 0.001, Opt: Adam)
Epoch 1/5 - Loss: 0.5660
Epoch 2/5 - Loss: 0.3679
Epoch 3/5 - Loss: 0.3384
Epoch 4/5 - Loss: 0.3228
Epoch 5/5 - Loss: 0.3105

Validation Accuracy after Feature Extraction: 0.8870


In [8]:
# 4. Conditional Fine-Tuning
fine_tuning_triggered = False
if val_acc < 0.80:
    print("\n[Phase 3] Validation < 80%. Unfreezing late layers (layer4) for fine-tuning...")
    fine_tuning_triggered = True

    # Unfreeze layer 4
    for param in model.layer4.parameters():
        param.requires_grad = True

    optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    model.train()

    for epoch in range(3): # Fine-tune for a few epochs
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer_ft.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_ft.step()
else:
    print("\n[Phase 3] Validation >= 80%. Fine-tuning bypassed.")


[Phase 3] Validation >= 80%. Fine-tuning bypassed.


In [9]:
# 5. Final Test Set Evaluation
print("\n[Phase 4] Evaluating on untouched Test Split...")
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

test_acc = accuracy_score(all_labels, all_preds)
print(f"Final Test Accuracy: {test_acc:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
print(cm)


[Phase 4] Evaluating on untouched Test Split...
Final Test Accuracy: 0.8818

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.80      0.86      0.83      1000
     Trouser       0.99      0.97      0.98      1000
    Pullover       0.80      0.89      0.84      1000
       Dress       0.86      0.89      0.87      1000
        Coat       0.78      0.83      0.80      1000
      Sandal       0.95      0.96      0.95      1000
       Shirt       0.77      0.56      0.65      1000
     Sneaker       0.94      0.93      0.93      1000
         Bag       0.98      0.98      0.98      1000
  Ankle boot       0.95      0.95      0.95      1000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


Confusion Matrix:
[[860   6  27  37   7   2  54   0   6   1]
 [  1 973   3  20   1   0   2   0   0   0]
 [ 16   0 892   6  56   0  29   0  

In [10]:
# 6. Save Artifacts & Export Images
torch.save(model.state_dict(), "models/product_classifier.pt")
print("\nModel saved to models/product_classifier.pt")

# Export 5 real images from raw dataset (no transforms applied so they save as proper images)
raw_test_data = datasets.FashionMNIST(root='./data', train=False, download=True)
exported_count = 0
for i in range(15):
    img, label_idx = raw_test_data[i]
    label_name = class_names[label_idx].replace('/', '_').lower()
    file_path = f"data/sample_images/{i}_{label_name}.png"
    img.save(file_path)
    exported_count += 1
    if exported_count >= 5:
        break

print("Exported 5 raw sample images to data/sample_images/")


Model saved to models/product_classifier.pt
Exported 5 raw sample images to data/sample_images/
